<a href="https://colab.research.google.com/github/haji8-de/AIFFEL_quest_rs/blob/main/DevelopResearch_2/langchain_kor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

last modified date : 2026.03.25  
제작 : 박광석 (모두의연구소)  
수정 : 하지양 gwd7454@gmail.com

# 랭체인으로 RAG 시작하기

해당 노트는 Langchain으로 RAG를 구현하기 위해 필요한
각 컴포넌트인 Document Loaders, Text splitters, Text embeddings, Vectorstores, Retriever를 다룹니다  




### Step 0 : 설치와 준비  
Langchain 설치 및 Gemini API 키를 등록하도록 합니다.  

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters


In [ ]:
import os


In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = ""

In [ ]:
! curl ipinfo.io

{
  "ip": "34.106.11.229",
  "hostname": "229.11.106.34.bc.googleusercontent.com",
  "city": "Salt Lake City",
  "region": "Utah",
  "country": "US",
  "loc": "40.7608,-111.8911",
  "org": "AS396982 Google LLC",
  "postal": "84101",
  "timezone": "America/Denver",
  "readme": "https://ipinfo.io/missingauth"
}

In [ ]:
from langchain_openai import ChatOpenAI

# OpenAI API를 사용하는 설정으로 변경
# 모델명은 필요에 따라 "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo" 등으로 바꿀 수 있습니다.
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
)

In [ ]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

### Step 1 : Document Loaders 사용해보기  

Document Loader는 다양한 형태의 원본 데이터를  
LLM이 이해할 수 있는 Document 객체(text + metadata) 로 변환하는 역할을 합니다.

PDF, 웹페이지, CSV와 같이 형식이 서로 다른 문서들을 일관된 구조로 파싱하여, 이후 Chunking·Embedding·검색(Retrieval) 단계에서
바로 사용할 수 있도록 만들어줍니다.

즉, Document Loader는
**RAG 파이프라인의 가장 첫 단계에서 “데이터를 읽을 수 있는 형태로 정리하는 역할을 담당**합니다.

공식 문서에서는 지원되는 다양한 Loader 목록을 확인할 수 있습니다.
https://python.langchain.com/docs/modules/data_connection/document_loaders/

#### PDFLoader 사용  
이번 실습에서는 가장 많이 사용되는 문서 형식인 PDF 파일을 대상으로
PyPDFLoader를 사용해 문서를 불러옵니다.

실습을 위해, 질의응답에 활용하고 싶은 PDF 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

PDFLoader는 각 페이지를 하나의 Document 단위로 변환하며,
이 단계에서 생성된 문서들은 이후 Text Splitter를 통해 의미 단위로 다시 분할됩니다.

In [ ]:
# from langchain_community.document_loaders import PyPDFLoader

# loader = PyPDFLoader("/content/Demian.pdf")
# pages = loader.load_and_split()

In [ ]:
# pages[0]

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [ ]:
# print(pages[10])

page_content='TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut F

출력 결과를 보기 쉽게 확인하기 위해,
Document 객체 전체가 아닌 실제 텍스트 본문이 담긴 page_content만 선택하여 확인해보겠습니다.

In [ ]:
# print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

#### CSVLoader

SV 파일은 행(row) 단위로 구조화된 데이터를 담고 있는 형식으로,
LangChain의 CSVLoader를 사용하면 각 행을 하나의 Document 객체로 변환할 수 있습니다.

이렇게 변환된 문서들은 이후 PDF나 웹 문서와 동일하게
Embedding, VectorStore, Retrieval 단계에서 함께 활용할 수 있습니다.

실습을 위해, CSV 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

In [ ]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("/content/drive/MyDrive/더하지/논문 작성/자료/제96회 TOPIK I (한국어능력시험)의 읽기 영역(31번~70번) 문제와 보기(정답 예시)의 사본 - 읽기 영역에 대한 문제, 정답 예시, 정답을 csv 또는 markdown으로 만들어줘.csv")

data = loader.load()

In [ ]:
data[:3]

[Document(metadata={'source': '/content/drive/MyDrive/더하지/논문 작성/자료/제96회 TOPIK I (한국어능력시험)의 읽기 영역(31번~70번) 문제와 보기(정답 예시)의 사본 - 읽기 영역에 대한 문제, 정답 예시, 정답을 csv 또는 markdown으로 만들어줘.csv', 'row': 0}, page_content='문제 번호: 31\n문제 및 지문: 저는 영국 사람입니다. 친구는 중국 사람입니다.\n보기 (정답 예시): ① 가족 ② 날짜 ③ 나라 ④ 학교\n정답: 3'),
 Document(metadata={'source': '/content/drive/MyDrive/더하지/논문 작성/자료/제96회 TOPIK I (한국어능력시험)의 읽기 영역(31번~70번) 문제와 보기(정답 예시)의 사본 - 읽기 영역에 대한 문제, 정답 예시, 정답을 csv 또는 markdown으로 만들어줘.csv', 'row': 1}, page_content='문제 번호: 32\n문제 및 지문: 민수 씨는 의사입니다. 병원에서 일합니다.\n보기 (정답 예시): ① 요일 ② 여행 ③ 취미 ④ 직업\n정답: 4'),
 Document(metadata={'source': '/content/drive/MyDrive/더하지/논문 작성/자료/제96회 TOPIK I (한국어능력시험)의 읽기 영역(31번~70번) 문제와 보기(정답 예시)의 사본 - 읽기 영역에 대한 문제, 정답 예시, 정답을 csv 또는 markdown으로 만들어줘.csv', 'row': 2}, page_content='문제 번호: 33\n문제 및 지문: 오늘은 비가 옵니다. 조금 춥습니다.\n보기 (정답 예시): ① 운동 ② 날씨 ③ 방학 ④ 계획\n정답: 2')]

#### 웹베이스로더  
웹베이스 로더는 웹페이지에 포함된 텍스트 콘텐츠를 직접 파싱하여 Document 객체로 변환하는 역할을 합니다.  
이를 통해 뉴스 기사, 블로그 글, 공지사항과 같은 실시간으로 업데이트되는 웹 문서를 RAG 시스템의 지식 소스로 활용할 수 있습니다.  
이번 실습에서는 실제 뉴스 기사를 예제로 사용하여,
웹페이지의 내용을 불러오고 텍스트 형태로 변환하는 과정을 살펴봅니다.  

실습에 사용할 웹페이지는 다음과 같습니다.  
https://it.chosun.com/news/articleView.html?idxno=2023092111831

In [ ]:
# from langchain_community.document_loaders import WebBaseLoader

In [ ]:
# loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
# documents = loader.load()

#print(documents[0].page_content)

주석을 해제하고 코드를 실행하면,
해당 웹페이지에 포함된 본문 텍스트 전체를 불러와 확인할 수 있습니다.  

웹페이지, PDF, CSV 등 서로 다른 형식의 문서들이
모두 텍스트 형태로 정상적으로 파싱된 것을 확인할 수 있습니다.  

이제 이 텍스트를 **전처리(불필요한 요소 제거, 정제)** 한 뒤,
Chunking과 Embedding 단계에 활용할 수 있습니다.  

### Step2 : TextSplitters 사용해보기  
Text Splitter는 긴 텍스트 문서를 **의미를 유지한 작은 단위(Chunk)** 로 분할하는 역할을 합니다.  
LLM은 한 번에 처리할 수 있는 토큰 수에 제한이 있기 때문에, 문서를 그대로 입력하는 대신 Splitter를 통해 분할된 여러 Chunk를 입력받아 처리하게 됩니다.  

이 과정을 통해 긴 문서에서도 토큰 길이 제약을 극복하고, 필요한 부분만 효율적으로 검색할 수 있습니다.  

분할된 각 Chunk는 이후 단계에서 1:1로 Embedding되어 VectorStore에 저장되며,
이 Chunk 단위가 RAG 시스템에서 검색과 응답의 기본 단위가 됩니다.  

In [ ]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

CharacterTextSplitter는
하나의 고정된 구분자(separator)를 기준으로 텍스트를 분할하는 방식입니다.
구현이 단순하고 직관적이지만,
문서 구조에 따라 분할된 Chunk가 토큰 제한을 초과하는 경우가 발생할 수 있습니다.

반면, RecursiveCharacterTextSplitter는
줄바꿈, 문장 구분자, 구두점 등 여러 구분자를 순차적으로 적용하며
텍스트를 재귀적으로 분할합니다.

이 방식은 토큰 제한을 안정적으로 만족시키는 데 유리하지만,
분할 과정에서 의미적으로 완전하지 않은 문장 단위로 잘릴 수 있다는 단점이 있습니다.  

단순한 구조의 문서나,
문단 구성이 명확한 텍스트의 경우에는 CharacterTextSplitter로도 충분합니다.

하지만 실제 서비스 환경에서는
문서 길이와 구조가 제각각인 경우가 많기 때문에,
대부분의 RAG 시스템에서는 RecursiveCharacterTextSplitter를 기본 선택지로 사용합니다.

이는 Chunk 크기를 안정적으로 제어하면서도
검색 실패를 줄이는 데 유리하기 때문입니다.

In [ ]:
with open("/content/drive/MyDrive/더하지/논문 작성/자료/제96회 TOPIK I (한국어능력시험)의 읽기 영역(31번~70번) 문제와 보기(정답 예시)의 사본 - 읽기 영역에 대한 문제, 정답 예시, 정답을 csv 또는 markdown으로 만들어줘.csv") as f:
    text = f.read()

In [ ]:
#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수입니다.
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정하는 것입니다.
text_splitter = CharacterTextSplitter(separator="\n", chunk_size=1000, chunk_overlap=100, length_function = len,)
chunks = text_splitter.split_text(text)

Chunk의 내용을 확인해보겠습니다

In [ ]:
print(chunks[0])

문제 번호,문제 및 지문,보기 (정답 예시),정답
31,저는 영국 사람입니다. 친구는 중국 사람입니다.,① 가족 ② 날짜 ③ 나라 ④ 학교,3
32,민수 씨는 의사입니다. 병원에서 일합니다.,① 요일 ② 여행 ③ 취미 ④ 직업,4
33,오늘은 비가 옵니다. 조금 춥습니다.,① 운동 ② 날씨 ③ 방학 ④ 계획,2
34,( )에 갑니다. 책을 삽니다.,① 은행 ② 극장 ③ 서점 ④ 식당,3
35,신발이 ( ). 발이 아픕니다.,① 같습니다 ② 좋습니다 ③ 작습니다 ④ 많습니다,3
36,오후에 약속이 있습니다. 민수 씨를 ( ).,① 찍습니다 ② 배웁니다 ③ 모릅니다 ④ 만납니다,4
37,이것은 제 가방이 아닙니다. 동생 ( ) 가방입니다.,① 도 ② 만 ③ 의 ④ 을,3
38,회사에 일이 많습니다. ( ) 바쁩니다.,① 빨리 ② 아까 ③ 먼저 ④ 아주,4
39,오늘은 제 생일입니다. 집에 친구들을 ( ).,① 줍니다 ② 말합니다 ③ 좋아합니다 ④ 초대합니다,4
40,"[1,000원의 아침밥] 안내문을 읽고 맞지 않는 것을 고르십시오.",① 천 원에 팝니다.② 구월까지 합니다.③ 주말에는 못 먹습니다.④ 학생 식당에서 먹습니다.,2
41,[KTX 승차권] 안내문을 읽고 맞지 않는 것을 고르십시오.,① 두 명이 갑니다.② 오전에 출발합니다.③ 인주에 세 시에 도착합니다.④ 시월 십이 일에 기차를 탑니다.,2
42,[메시지 대화] 다음을 읽고 맞지 않는 것을 고르십시오.,① 지영 씨는 바빴습니다.② 수미 씨는 지영 씨를 만날 겁니다.③ 수미 씨는 졸업 선물을 샀습니다.④ 지영 씨는 선물을 사러 갈 겁니다.,3
43,우리 집 근처에 빵집이 하나 있습니다. 그 빵집은 케이크가 싸고 맛있습니다. 저는 케이크를 좋아해서 거기에 자주 갑니다.,① 빵집이 집에서 멉니다.② 저는 빵집에 자주 갑니다.③ 저는 케이크를 싫어합니다.④ 케이크가 비싸지만 맛있습니다.,2


각 chunk의 길이를 확인해보겠습니다,

In [ ]:
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)

[933, 943, 868, 967, 724]


### 토큰 단위로 텍스트 분할해보기  
  
LLM은 문장을 단어가 아닌 토큰(token) 단위로 처리합니다.
따라서 사람이 인식하는 단어 길이나 문자 수는
실제 모델이 처리하는 입력 길이와 정확히 일치하지 않을 수 있습니다.

이로 인해 문자 수나 단어 수를 기준으로 텍스트를 분할할 경우,
모델의 입력 토큰 제한을 초과하거나
예상보다 훨씬 짧은 문맥만 전달되는 문제가 발생할 수 있습니다.

실제 서비스 환경에서는 이러한 문제를 방지하기 위해,
토큰 단위를 기준으로 텍스트를 분할하는 방식을 사용합니다.
이제 토큰 기준으로 텍스트를 분할해보겠습니다.

In [ ]:
!pip install tiktoken

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

In [ ]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

[933, 943, 868, 967, 724]
[885, 881, 790, 901, 670]


글자 수와 토큰 수의 차이를 확인할 수 있습니다 !

### Step3 : TextEmbedding 사용해보기  
Embedding은 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터(vector) 형태로 변환하는 과정입니다.
이 벡터는 문장의 표면적인 형태가 아니라, 의미적 유사성을 반영하도록 설계되어 있습니다.

변환된 벡터는
VectorStore에 저장되거나,
새로운 질의(Query) 벡터와의 유사도 계산을 통해
의미적으로 가까운 문서를 검색하는 데 사용됩니다.

이러한 변환은 대규모 말뭉치로 사전 학습된
Embedding 전용 모델을 통해 이루어지며,
RAG 시스템에서 Retrieval 성능을 결정하는 핵심 요소입니다.

이번 실습에서는
OpenAI 임베딩 모델을 사용해
텍스트를 벡터로 변환해보겠습니다.

In [ ]:
import openai

genai 라이브러리의 list_models 함수를 사용하여 사용 가능한 모델들의 목록을 가져옵니다.

In [ ]:
client = openai.OpenAI()

In [ ]:
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

text-embedding-3-small
text-embedding-3-large
text-embedding-ada-002


text-embedding-3-small은 가성비가 좋고, text-embedding-3-large는 성능이 더 강력합니다.

In [ ]:
from langchain_openai import OpenAIEmbeddings


embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

만약 여러분들이 Gemini를 사용하여 구축중이시라면, embedding은 지역에 따라 사용이 제한됩니다.  
주로 유럽권에서 제한되기 때문에, 다음 에러를 확인하신다면 Colab 파일의 서버 저장 위치를 확인 후, 다른 임베딩 모델로 변경해야합니다.  

Error embedding content: 400 User location is not supported for the API use.


In [ ]:
!curl ipinfo.io

{
  "ip": "34.106.11.229",
  "hostname": "229.11.106.34.bc.googleusercontent.com",
  "city": "Salt Lake City",
  "region": "Utah",
  "country": "US",
  "loc": "40.7608,-111.8911",
  "org": "AS396982 Google LLC",
  "postal": "84101",
  "timezone": "America/Denver",
  "readme": "https://ipinfo.io/missingauth"
}

In [ ]:
# 400 User location is not supported for the API use 오류가 발생한다면, 이 블록을 대신 실행해주세요

# ! pip install -q sentence_transformers

#from langchain.embeddings import HuggingFaceEmbeddings
#embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

embedding model 변수에 OpenAI 임베딩모델 혹은 huggingface의 임베딩모델이 할당되었을 것입니다.  
embed_documents 멤버 함수를 사용하여 새 문장을 변환해보겠습니다  

In [ ]:
embeddings = embedding_model.embed_documents(
    [
        "저는 영국 사람입니다. 친구는 중국 사람입니다",
        "민수 씨는 의사입니다. 병원에서 일합니다.",
        "오늘은 비가 옵니다. 조금 춥습니다.",
    ]
)

임베딩으로 잘 변환되었는지 확인해보겠습니다  

In [ ]:
print(embeddings[0])

[-0.031146544963121414, 0.007047045510262251, -0.07621042430400848, 0.021490000188350677, -0.046961694955825806, 0.0019943369552493095, -3.739382009371184e-05, 0.030197635293006897, -0.04874787852168083, -0.026122909039258957, -0.027518363669514656, 0.02016896940767765, -0.01383360568434, 0.038440119475126266, 0.03564921021461487, 0.011526454240083694, -0.04625466465950012, 0.02191793918609619, -0.00869368202984333, 0.019275877624750137, 0.0070237875916063786, 0.009042545221745968, 0.05351102724671364, -0.033900242298841476, -0.01859675720334053, -0.02137836255133152, 0.03369557484984398, -0.01667102985084057, -0.027592787519097328, -0.013033545576035976, 0.024932119995355606, -0.026160120964050293, -0.036319028586149216, -0.022438907995820045, 0.02100624144077301, 0.0017757158493623137, -0.013191696256399155, -0.01748969592154026, -0.0002005965798161924, -0.0062004695646464825, 8.525355224264786e-05, -0.04618024080991745, 0.032262906432151794, -0.017601333558559418, -0.034607272595167

In [ ]:
len(embeddings[0])

1536

새로운 쿼리를 넣어, 임베딩끼리 유사도를 계산해보겠습니다

In [ ]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [ ]:
query = ["저는 한국 사람입니다."]

In [ ]:
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

NameError: name 'query' is not defined

빨간 사과와 빨간 과일의 유사도가 많이 높게 나왔습니다!  
  
임베딩 모델은 사용 언어나 필요에 따라 다양하게 교체하여 사용할 수 있습니다.  
해당 링크에서 여러 목록을 확인하실 수 있습니다.  
https://python.langchain.com/docs/integrations/text_embedding/

### Step4 : VectorStore 사용해보기
VectorStore는 텍스트를 Embedding 모델을 통해 벡터(vector)로 변환한 뒤, 이를 저장하고 관리하는 저장소입니다.
이 저장소는 단순한 데이터 보관 공간이 아니라,
벡터 간의 유사도를 빠르게 계산하고 탐색하기 위한 인덱싱 구조를 함께 포함하고 있습니다.

문서나 쿼리가 Embedding된 이후에는,
VectorStore를 통해 의미적으로 유사한 벡터를 효율적으로 검색할 수 있으며,
이 과정이 RAG 시스템의 Retrieval 단계를 담당하게 됩니다.

대표적인 VectorStore로는
Chroma, FAISS 등이 있으며,
각각 로컬 환경과 대규모 서비스 환경에서 널리 사용됩니다.

이번 실습에서는
구성이 단순하고 로컬 환경에서 바로 사용할 수 있는
ChromaDB를 사용해 VectorStore를 구성해보겠습니다.

In [ ]:
!pip install chromadb

In [ ]:
!pip install langchain-chroma

In [ ]:
from langchain_chroma import Chroma

In [ ]:
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk

제일 처음에 사용했던, PDF를 다시 사용하도록 합니다!  

In [ ]:
# 위에서 사용했던 코드입니다
# loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

In [ ]:
!pip show chromadb

Name: chromadb
Version: 1.5.5
Summary: Chroma.
Home-page: https://github.com/chroma-core/chroma
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, pybase64, pydantic, pydantic-settings, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: 


Chroma에 임베딩 시킵니다  

In [ ]:
db = Chroma.from_documents(docs, embedding_model)


이제 쿼리를 날려보겠습니다

In [ ]:
query = "민수씨의 직업은 무엇입니까?"
docs = db.similarity_search(query)

In [ ]:
print(docs[0].page_content)

문제 번호: 32
문제 및 지문: 민수 씨는 의사입니다. 병원에서 일합니다.
보기 (정답 예시): ① 요일 ② 여행 ③ 취미 ④ 직업
정답: 4


Face, features, looks like 등 데미안의 생김새를 담고 있는 페이지가 출력되었습니다  
굉장히 빠른 속도로 검색했습니다!  

### Step5 : Retriever 사용해보기  

Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤,
VectorStore에 저장된 문서 벡터들과 비교하여
의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할을 합니다.

즉, Retriever는
RAG 시스템에서 “어떤 정보를 LLM에게 참고 자료로 줄 것인가”를 결정하는 핵심 컴포넌트이며,
검색 결과의 품질이 곧 최종 답변의 품질로 이어집니다.
  

In [ ]:
!pip install -U langchain langchain-classic

In [ ]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA


긴 문서 전체를 한 번에 LLM에 전달하는 대신,
Retriever와 LLM을 결합한 RetrievalQA 체인을 사용하여
문서에서 질문과 관련된 부분만 검색하고,
그 결과를 바탕으로 답변을 생성합니다.

이를 통해 길이가 긴 문서에서도
토큰 제한을 넘지 않으면서, 근거 기반의 질의응답을 수행할 수 있습니다.

In [ ]:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# OpenAI 모델로 변경
# streaming=True와 callbacks 설정을 통해 실시간 출력을 활성화합니다.
llm = ChatOpenAI(
    model="gpt-4o",              # 또는 "gpt-4o-mini"
    temperature=0.0,
    streaming=True,              # 실시간 출력을 켭니다
    callbacks=[StreamingStdOutCallbackHandler()], # 출력을 콘솔에 바로 뿌려줍니다
)

체인의 종류와 검색(Retrieval) 방식,
그리고 그에 따른 주요 파라미터를 설정합니다.

이 단계에서는
Retriever가 어떤 전략으로 문서를 검색할지,
그리고 몇 개의 문서를 LLM에게 전달할지를 결정하게 됩니다.
이 선택은 최종 답변의 품질과 직접적으로 연결됩니다.

예를 들어,
MMR(Maximal Marginal Relevance) 방식은
쿼리와의 유사도뿐만 아니라 문서 간의 중복을 줄이고 다양성을 확보하는 재정렬(Re-ranking) 전략입니다.

실무 환경에서는 단일 문서에 정보가 몰리는 것을 방지하고,
LLM이 보다 풍부한 문맥을 참고하도록 하기 위해
MMR 방식이 자주 사용됩니다.

In [ ]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

위 코드에서 짚고 넘어갈 파라미터는 다음과 같습니다  
🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다.
구조가 단순하고 이해하기 쉬워,
RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다.
단점으로는 문서 수가 많아질 경우
토큰 사용량이 빠르게 증가할 수 있습니다.
실무에서는 초기 검증 단계에서는 stuff를,
문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.  

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다.
검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.  

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}  
- fetch_k  
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
- k  
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.

일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.  

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.

In [ ]:
query = "문제번호 32번 '민수 씨는 의사입니다. 병원에서 일합니다.' 문제를 틀렸을때 문맥적으로 동일한 문제를 다시 제출하려면 몇번 문제를 제출해야합니까?"
result = qa(query)

문제 번호 32번과 문맥적으로 동일한 문제는 '직업'에 관한 문제입니다. 따라서, 동일한 주제를 다루는 문제는 문제 번호 35번입니다.

마크다운 형식으로 출력해봅니다

In [ ]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

문제 번호 32번과 문맥적으로 동일한 문제는 '직업'에 관한 문제입니다. 따라서, 동일한 주제를 다루는 문제는 문제 번호 35번입니다.

RAG를 사용하지 않은 llm 호출도 시도해보세요!

In [ ]:
llm2 = ChatOpenAI(
    model="gpt-4o")
request = llm2.invoke("문제번호 32번 '민수 씨는 의사입니다. 병원에서 일합니다.' 문제를 틀렸을때 문맥적으로 동일한 문제를 다시 제출하려면 몇번 문제를 제출해야합니까")
display(Markdown(request.content))


주어진 문제와 비슷한 문맥의 문제를 다시 제출하려면, 새로운 문제번호로 문제를 생성하여 제출해야 합니다. 이미 있는 문제를 다른 번호로 다시 제출하는 기능은 일반적으로 제공되지 않습니다. 따라서, 동일한 문맥의 새로운 문제를 작성하여 새로운 문제번호로 제출하는 방법을 고려해야 합니다. 문제 제출 시스템의 세부 절차는 해당 플랫폼이나 시스템의 규정을 참조해야 정확하게 알 수 있습니다.

### Quiz
결과의 어떤 부분을 관찰하였을 때, RAG 시스템의 결과를 신뢰할 수 있겠다 생각하셨나요?  

### Answer  
원문에서 답변의 출처를 확인할 수 있었습니다.

## 6. 완성 예제  
앞에서 진행한 내용으로, Demian을 다시 한번 읽어봅시다!  
완성하여 제출해주세요~


필요한 라이브러리를 모두 다운받습니다  

Text splitter 사용을 위한 준비입니다

### Step 1 Document loader

### Step 2 Text splitters

### Step 3 Vector Empeddings

### Step 4 Retrievers

### Step 5 Question Answering